In [1]:
# Instala versão do kafka para Python 3.11+
%pip install kafka-python-ng boto3 pillow --quiet

from kafka import KafkaConsumer
import json, time, boto3
from datetime import datetime

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# Configura o cliente do MinIO (Simulando o AWS S3)
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='root-minio',
    aws_secret_access_key='root12345678',
    region_name='us-east-1'
)

consumer = KafkaConsumer(
    'cdc.public.transacoes_financeiras',
    bootstrap_servers=['kafka:29092'],
    auto_offset_reset='earliest',
    enable_auto_commit=True,
    auto_commit_interval_ms=5000,  # commita a cada 5 segundos
    group_id='grupo-consumer-v1',
    value_deserializer=lambda x: json.loads(x.decode('utf-8'))
)

print("Conexões estabelecidas! Pronto para consumir...")

Conexões estabelecidas! Pronto para consumir...


In [ ]:
buffer = []
ULTIMO_FLUSH = time.time()

for msg in consumer:
    envelope = msg.value
    payload = envelope.get('payload', envelope)
    operacao = payload.get('op')
    
    print(f"MSG recebida | offset={msg.offset} | op={operacao}")  # debug
    
    if operacao in ['c', 'u', 'r']:
        dados_tx = payload.get('after')
        if dados_tx:
            dados_tx['tipo_operacao_cdc'] = 'SNAPSHOT' if operacao == 'r' else 'INSERT' if operacao == 'c' else 'UPDATE'
            buffer.append(dados_tx)
    elif operacao == 'd':
        dados_tx = payload.get('before')
        if dados_tx:
            dados_tx['tipo_operacao_cdc'] = 'AUDITORIA_DELETE'
            buffer.append(dados_tx)

    if len(buffer) >= 5 or (time.time() - ULTIMO_FLUSH >= 10 and len(buffer) > 0):
        
        timestamp_str = datetime.now().strftime("%Y%m%d-%H%M%S")
        nome_arquivo = f"streaming/financeiro/tx_batch_{timestamp_str}.jsonl"
        conteudo_jsonl = "\n".join([json.dumps(m) for m in buffer])
        s3_client.put_object(Bucket="raw", Key=nome_arquivo, Body=conteudo_jsonl.encode('utf-8'))
        consumer.commit() 
        
        print(f"[FLUSH] {len(buffer)} registros → {nome_arquivo}")
        
        buffer.clear()
        ULTIMO_FLUSH = time.time()